In [9]:
import pandas as pd
import numpy as np
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [2]:
with open("api_key.txt", "r") as file:
    api_key = file.read().strip()

In [3]:
data = pd.read_csv("healthcare_dataset.csv", encoding="utf-8")

In [4]:
data

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,eLIZABeTH jaCkSOn,42,Female,O+,Asthma,2020-08-16,Joshua Jarvis,Jones-Thompson,Blue Cross,2650.714952,417,Elective,2020-09-15,Penicillin,Abnormal
55496,KYle pEREz,61,Female,AB-,Obesity,2020-01-23,Taylor Sullivan,Tucker-Moyer,Cigna,31457.797307,316,Elective,2020-02-01,Aspirin,Normal
55497,HEATher WaNG,38,Female,B+,Hypertension,2020-07-13,Joe Jacobs DVM,"and Mahoney Johnson Vasquez,",UnitedHealthcare,27620.764717,347,Urgent,2020-08-10,Ibuprofen,Abnormal
55498,JENniFER JOneS,43,Male,O-,Arthritis,2019-05-25,Kimberly Curry,"Jackson Todd and Castro,",Medicare,32451.092358,321,Elective,2019-05-31,Ibuprofen,Abnormal


In [5]:
docs  = []
embed_list = []
content_cols = [col for col in data.columns]
for index, row in data.iterrows():
    page_content = "\n".join([f"{col}: {row[col]}" for col in content_cols])
    embed_list.append(page_content)
    doc = Document(page_content=page_content)
    docs.append(doc)

In [6]:
model = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5084.04it/s]


In [7]:
vector_store = Chroma.from_documents(
    documents=docs[:10],
    embedding=model,
    persist_directory="./chroma_langchain_db"
)

In [8]:
query = "emergency and cell mutation"
docs = vector_store.similarity_search(query, k=1)

print("\n--- Result ---")
print(docs[0].page_content)


--- Result ---
Name: adrIENNE bEll
Age: 43
Gender: Female
Blood Type: AB+
Medical Condition: Cancer
Date of Admission: 2022-09-19
Doctor: Kathleen Hanna
Hospital: White-White
Insurance Provider: Aetna
Billing Amount: 14238.317813937623
Room Number: 458
Admission Type: Urgent
Discharge Date: 2022-10-09
Medication: Penicillin
Test Results: Abnormal


In [14]:
context = docs[0].page_content

llm = ChatGroq(
    api_key=api_key,
    model="qwen/qwen3-32b",
    temperature=0.2
)
messages = [
    (
        "system",
        f"You are a helpful assistant that answer the question. Here is context {context}",
    ),
    ("human", "is the patient child, teen, or adult? and what's her bloodtype?"),
]
ai_msg = llm.invoke(messages)
print(ai_msg.content)

<think>
Okay, let's see. The user is asking if the patient is a child, teen, or adult, and what her blood type is. Let me check the provided information.

First, the patient's name is AdrIenne Bell, and her age is 43. Since 43 is well into adulthood, the answer for the first part should be adult. Then, the blood type is listed as AB+. I need to make sure I'm reading that correctly. The medical condition is cancer, but that's not relevant here. The blood type is clearly AB+. 

Wait, the user might be confused about the age categories. Let me confirm: children are typically under 12, teens between 13-19, and adults 20 and above. At 43, AdrIenne is definitely an adult. The blood type is straightforward from the data given. No need to overcomplicate. Just state the age category and the blood type as provided.
</think>

The patient, AdrIenne Bell, is an **adult** (age 43). Her blood type is **AB+**.
